# NUS ST4238 — Stochastic Processes II
## A detailed, intuitive, simulation-driven learning notebook

This notebook is designed as a **step-by-step companion** for advanced undergraduate study of continuous-time stochastic processes.

The main learning philosophy is:

> **Story → state variable → assumptions → mathematical model → simulation → visualisation → interpretation → limitations.**

Rather than memorising isolated formulas, we will repeatedly ask:

1. **What is random?**
2. **What evolves through time?**
3. **What is the state of the system?**
4. **Which events may occur?**
5. **At what rates do they occur?**
6. **How does each event modify the state?**
7. **What can be computed analytically?**
8. **What should be simulated?**
9. **How do we check whether our simulation agrees with theory?**

### Topics covered

1. Poisson processes
2. Exponential waiting times and arrival epochs
3. Superposition and thinning
4. Non-homogeneous Poisson processes
5. Compound Poisson processes
6. Continuous-time Markov chains (CTMCs)
7. Generator matrices and Kolmogorov equations
8. Birth-death processes
9. $M/M/1$ and $M/M/c$ queues
10. Stochastic epidemic models
11. Renewal processes
12. Renewal-reward processes
13. Martingales, submartingales and supermartingales
14. Stopping times and optional-stopping intuition
15. Brownian motion
16. Drifted Brownian motion
17. Quadratic variation
18. First-passage / hitting times
19. Integrated case studies and revision exercises

> **Syllabus note.** Exact semester emphasis can vary by lecturer. This notebook deliberately covers the core/recent themes associated with ST4238 and also the classical renewal/Brownian-motion material that has appeared in historical treatments of the module.

# 0. How the major ideas fit together

A useful conceptual map is:

$$
\text{random arrivals}
\rightarrow
\text{Poisson process}
\rightarrow
\text{exponential holding times}
\rightarrow
\text{CTMC}
\rightarrow
\text{birth-death processes}
\rightarrow
\text{queues / epidemics / reliability}
$$

Then we generalise in two directions:

$$
\text{exponential inter-arrivals}
\rightarrow
\text{general inter-arrivals}
\rightarrow
\text{renewal processes},
$$

and

$$
\text{discrete random fluctuations}
\rightarrow
\text{martingales}
\rightarrow
\text{Brownian motion}.
$$

The unifying idea is that **randomness is indexed by time**, and we care about both:

- **transient behaviour**: what happens at finite time $t$;
- **long-run behaviour**: what happens as $t\to\infty$.

# 1. Environment and reproducibility

The notebook uses:

- `numpy` for simulation,
- `pandas` for summaries,
- `scipy` for probability distributions and matrix exponentials,
- `bokeh` for all visualisations.

The random seed is fixed so that plots and numerical experiments are reproducible.

In [1]:
# If needed, uncomment the next line in a fresh environment:
# %pip install numpy pandas scipy bokeh

from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Iterable, Optional
import math

import numpy as np
import pandas as pd

from scipy.linalg import expm
from scipy.special import gammaln
from scipy.stats import poisson, expon, gamma, norm

from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    DataTable,
    TableColumn,
    NumberFormatter,
    Div,
)
from bokeh.palettes import Category10
from bokeh.plotting import figure

output_notebook()

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 4)

Loading BokehJS ...

## 1.1 Reusable plotting helpers

A recurring source of notebook clutter is repeated plotting code.  
We therefore define small reusable helpers.

This is deliberately simple rather than over-engineered: the goal is for the statistical model to remain visible.

In [2]:
PALETTE = Category10[10]

def line_plot(
    x,
    ys: dict[str, np.ndarray],
    *,
    title: str,
    x_label: str,
    y_label: str,
    width: int = 760,
    height: int = 390,
    legend_location: str = "top_left",
):
    p = figure(
        title=title,
        width=width,
        height=height,
        x_axis_label=x_label,
        y_axis_label=y_label,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    for idx, (label, y) in enumerate(ys.items()):
        color = PALETTE[idx % len(PALETTE)]
        p.line(x, y, line_width=2.3, color=color, legend_label=label)

    if p.legend:
        p.legend.location = legend_location
        p.legend.click_policy = "hide"
    return p


def scatter_plot(
    x,
    y,
    *,
    title: str,
    x_label: str,
    y_label: str,
    hover: Optional[list[tuple[str, str]]] = None,
    width: int = 760,
    height: int = 390,
):
    source = ColumnDataSource({"x": x, "y": y})
    p = figure(
        title=title,
        width=width,
        height=height,
        x_axis_label=x_label,
        y_axis_label=y_label,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p.scatter("x", "y", source=source, size=7, alpha=0.7)
    if hover:
        p.add_tools(HoverTool(tooltips=hover))
    return p


def bokeh_table(df: pd.DataFrame, *, width: int = 760, height: int = 250):
    safe = df.copy().reset_index(drop=True)
    source = ColumnDataSource(safe)

    columns = []
    for c in safe.columns:
        if pd.api.types.is_numeric_dtype(safe[c]):
            columns.append(
                TableColumn(
                    field=str(c),
                    title=str(c),
                    formatter=NumberFormatter(format="0.0000"),
                )
            )
        else:
            columns.append(TableColumn(field=str(c), title=str(c)))

    return DataTable(
        source=source,
        columns=columns,
        width=width,
        height=height,
        index_position=None,
    )

# 2. Poisson processes — random events occurring in continuous time

Suppose $N(t)$ counts how many events have occurred by time $t$.

Examples:

- calls reaching a service desk;
- claims arriving at an insurer;
- jobs reaching a server;
- component failures;
- mutations;
- transactions;
- packet arrivals.

A **homogeneous Poisson process** with rate $\lambda>0$ satisfies:

1. $N(0)=0$;
2. increments over disjoint time intervals are independent;
3. increments depend only on interval length;
4. for small $h$,

$$
P(N(t+h)-N(t)=1)=\lambda h+o(h),
$$

while two or more events in a tiny interval have probability $o(h)$.

The count at time $t$ is

$$
N(t)\sim\operatorname{Poisson}(\lambda t).
$$

Hence

$$
P(N(t)=k)
=
e^{-\lambda t}\frac{(\lambda t)^k}{k!}.
$$

The rate $\lambda$ has units of **events per unit time**.

## 2.1 Simulating a Poisson process through inter-arrival times

A very useful equivalence is:

$$
N(t)	ext{ is Poisson}
\iff
T_1,T_2,\ldots \overset{iid}{\sim}\operatorname{Exponential}(\lambda),
$$

where $T_i$ is the waiting time between consecutive arrivals.

So a simulation algorithm is:

1. sample an exponential waiting time;
2. advance the clock;
3. record the new arrival;
4. repeat until the horizon is exceeded.

If $m$ events occur, the algorithm is $\Theta(m)$.

In [36]:
@dataclass
class PoissonPath:
    rate: float

    def simulate_arrivals(
        self,
        horizon: float,
        rng: np.random.Generator,
    ) -> np.ndarray:
        arrivals = []
        t = 0.0

        while True:
            t += rng.exponential(1.0 / self.rate)
            if t > horizon:
                break
            arrivals.append(t)

        return np.asarray(arrivals, dtype=float)

    def step_path(
        self,
        horizon: float,
        rng: np.random.Generator,
    ) -> tuple[np.ndarray, np.ndarray]:
        arrivals = self.simulate_arrivals(horizon, rng)

        x = [0.0]
        y = [0]

        count = 0
        for t in arrivals:
            x.extend([t, t])
            y.extend([count, count + 1])
            count += 1

        x.append(horizon)
        y.append(count)

        return np.asarray(x), np.asarray(y)

In [37]:
lambda_rate = 4.0       # events/hour
horizon = 5.0           # hours
n_paths = 5

p = figure(
    title="Five simulated Poisson-process sample paths",
    width=800,
    height=420,
    x_axis_label="Time",
    y_axis_label="Cumulative number of arrivals",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

for i in range(n_paths):
    x, y = PoissonPath(lambda_rate).step_path(horizon, rng)
    p.line(
        x,
        y,
        line_width=2,
        alpha=0.85,
        color=PALETTE[i],
        legend_label=f"Path {i+1}",
    )

p.legend.location = "top_left"
p.legend.click_policy = "hide"
show(p)

### Interpretation

Every trajectory is a **step function**:

- time passes horizontally;
- the count jumps by one whenever an event occurs;
- jumps occur at random times.

Even though every path looks different, the expected slope is approximately

$$
E[N(t)] = \lambda t.
$$

So with $\lambda=4$, the process accumulates about four events per hour **on average**, not exactly.

## 2.2 Monte Carlo validation of the Poisson count distribution

Simulation should not replace theory; it should **test our understanding of theory**.

We simulate many independent values of $N(t)$ and compare the empirical probabilities with

$$
P(N(t)=k).
$$

In [38]:
lambda_rate = 3.0
t_eval = 2.0
n_sim = 50_000

sim_counts = rng.poisson(lambda_rate * t_eval, size=n_sim)

k_values = np.arange(0, int(np.percentile(sim_counts, 99.9)) + 2)
empirical = np.array([(sim_counts == k).mean() for k in k_values])
theoretical = poisson.pmf(k_values, mu=lambda_rate * t_eval)

source = ColumnDataSource(
    {
        "k": k_values,
        "empirical": empirical,
        "theoretical": theoretical,
    }
)

p = figure(
    title="Poisson counts: Monte Carlo frequencies versus exact PMF",
    width=800,
    height=420,
    x_axis_label="k",
    y_axis_label="Probability",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p.vbar(
    x="k",
    top="empirical",
    source=source,
    width=0.75,
    alpha=0.45,
    legend_label="Empirical",
)

p.scatter(
    x="k",
    y="theoretical",
    source=source,
    size=9,
    marker="circle",
    color=PALETTE[1],
    legend_label="Theoretical",
)

p.line(
    x="k",
    y="theoretical",
    source=source,
    line_width=2,
    color=PALETTE[1],
)

p.legend.location = "top_right"
show(p)

If the simulation is implemented correctly, the bars and theoretical points should nearly coincide.

The Monte Carlo estimation error typically decreases at rate

$$
O(n^{-1/2}),
$$

so making a simulation 100 times larger usually reduces standard error by only about 10 times.

## 2.3 Exponential waiting times and memorylessness

For a Poisson process with rate $\lambda$,

$$
T\sim\operatorname{Exponential}(\lambda),
$$

and

$$
P(T>t)=e^{-\lambda t}.
$$

The mean waiting time is

$$
E[T]=\frac{1}{\lambda}.
$$

The exponential distribution is memoryless:

$$
P(T>s+t\mid T>s)=P(T>t).
$$

This property is not merely a curiosity. It is the mathematical reason exponential holding times naturally produce **continuous-time Markov chains**.

In [41]:
lambda_rate = 2.5
n = 40_000
waits = rng.exponential(1 / lambda_rate, size=n)

hist, edges = np.histogram(waits, bins=45, density=True)
centers = 0.5 * (edges[:-1] + edges[1:])

grid = np.linspace(0, np.quantile(waits, 0.995), 400)
pdf = expon.pdf(grid, scale=1 / lambda_rate)

p = figure(
    title="Exponential inter-arrival times",
    width=800,
    height=420,
    x_axis_label="Waiting time",
    y_axis_label="Density",
)

p.vbar(
    x=centers,
    top=hist,
    width=np.diff(edges) * 0.92,
    alpha=0.45,
    legend_label="Simulation",
)

p.line(
    grid,
    pdf,
    line_width=3,
    color=PALETTE[1],
    legend_label="Exact exponential density",
)

p.legend.location = "top_right"
show(p)

print(f"Empirical mean waiting time: {waits.mean():.4f}")
print(f"Theoretical mean waiting time: {1/lambda_rate:.4f}")

Empirical mean waiting time: 0.3999
Theoretical mean waiting time: 0.4000


## 2.4 Arrival epochs and the Gamma distribution

The time of the $n$-th arrival is

$$
S_n=T_1+\cdots+T_n.
$$

Because the $T_i$ are exponential,

$$
S_n\sim\operatorname{Gamma}(n,\lambda)
$$

when we use the **shape-rate** parameterisation.

This gives the useful equivalence:

$$
N(t)\ge n
\iff
S_n\le t.
$$

One viewpoint counts events; the other studies waiting time until a target event number.

In [42]:
lambda_rate = 2.0
n_arrival = 5
n_sim = 30_000

arrival_epochs = rng.gamma(shape=n_arrival, scale=1/lambda_rate, size=n_sim)

hist, edges = np.histogram(arrival_epochs, bins=50, density=True)
centers = (edges[:-1] + edges[1:]) / 2

grid = np.linspace(0, np.quantile(arrival_epochs, 0.997), 500)
pdf = gamma.pdf(grid, a=n_arrival, scale=1/lambda_rate)

p = figure(
    title=f"Distribution of time to the {n_arrival}-th arrival",
    width=800,
    height=420,
    x_axis_label="Arrival epoch",
    y_axis_label="Density",
)

p.vbar(x=centers, top=hist, width=np.diff(edges)*0.92, alpha=0.45)
p.line(grid, pdf, line_width=3, color=PALETTE[1], legend_label="Gamma density")
p.legend.location = "top_right"

show(p)

# 3. Superposition and thinning

Poisson processes have unusually elegant closure properties.

## Superposition

If independent processes have rates $\lambda_1$ and $\lambda_2$, then

$$
N_1(t)+N_2(t)
$$

is Poisson with rate

$$
\lambda_1+\lambda_2.
$$

## Thinning

If every event in a rate-$\lambda$ Poisson process is independently labelled type A with probability $p$, then type-A events form a Poisson process with rate

$$
p\lambda.
$$

The remaining events form an independent Poisson process with rate

$$
(1-p)\lambda.
$$

This makes Poisson models easy to compose and decompose.

In [43]:
lambda_total = 12.0
p_api = 0.25
horizon = 1.0
n_sim = 50_000

total = rng.poisson(lambda_total * horizon, size=n_sim)
api = rng.binomial(total, p_api)
other = total - api

summary = pd.DataFrame({
    "stream": ["total", "API after thinning", "other after thinning"],
    "empirical_mean": [total.mean(), api.mean(), other.mean()],
    "theoretical_mean": [
        lambda_total * horizon,
        lambda_total * p_api * horizon,
        lambda_total * (1-p_api) * horizon,
    ],
    "empirical_variance": [
        total.var(ddof=1),
        api.var(ddof=1),
        other.var(ddof=1),
    ],
    "theoretical_variance": [
        lambda_total * horizon,
        lambda_total * p_api * horizon,
        lambda_total * (1-p_api) * horizon,
    ],
})

display(summary.round(4))
show(bokeh_table(summary.round(4), height=170))

,stream,empirical_mean,theoretical_mean,empirical_variance,theoretical_variance
0,total,11.9781,12.0,11.9583,12.0
1,API after thinning,2.9906,3.0,2.9889,3.0
2,other after thinning,8.9875,9.0,9.0160,9.0


### What should you notice?

For a Poisson random variable,

$$
E[N]=\operatorname{Var}(N).
$$

After thinning, the same mean-variance identity still appears for each stream.

This is a useful diagnostic: if real arrival data has variance far larger than the mean, a simple homogeneous Poisson model may be too restrictive.

# 4. Non-homogeneous Poisson processes

A constant event rate is often unrealistic.

A web service may have:

- low traffic overnight,
- rising traffic in the morning,
- a lunch peak,
- an evening decline.

We therefore allow a time-dependent intensity

$$
\lambda(t).
$$

The cumulative intensity is

$$
\Lambda(t)=\int_0^t \lambda(s)\,ds.
$$

Then

$$
N(t)\sim\operatorname{Poisson}(\Lambda(t)).
$$

The instantaneous rate tells us **how quickly expected event accumulation changes with time**.

In [44]:
def nhpp_thinning(
    intensity: Callable[[float], float],
    lambda_max: float,
    horizon: float,
    rng: np.random.Generator,
) -> np.ndarray:
    '''
    Lewis-Shedler thinning simulation for a non-homogeneous Poisson process.
    '''
    arrivals = []
    t = 0.0

    while True:
        t += rng.exponential(1.0 / lambda_max)
        if t > horizon:
            break

        accept_prob = intensity(t) / lambda_max
        if rng.uniform() < accept_prob:
            arrivals.append(t)

    return np.asarray(arrivals)


def traffic_intensity(t: float) -> float:
    # Smooth daily pattern over t in [0, 24].
    return 6.0 + 12.0 * np.exp(-0.5*((t-9)/2.0)**2) + 18.0 * np.exp(-0.5*((t-18)/2.5)**2)


grid = np.linspace(0, 24, 600)
lam_grid = np.array([traffic_intensity(t) for t in grid])
lambda_max = lam_grid.max() * 1.02

arrivals = nhpp_thinning(traffic_intensity, lambda_max, 24, rng)

p1 = figure(
    title="Time-varying event intensity",
    width=800,
    height=320,
    x_axis_label="Hour",
    y_axis_label="λ(t)",
)
p1.line(grid, lam_grid, line_width=3)

counts, edges = np.histogram(arrivals, bins=np.arange(0, 25, 1))
centers = (edges[:-1] + edges[1:]) / 2

p2 = figure(
    title="One simulated day of NHPP arrivals",
    width=800,
    height=320,
    x_axis_label="Hour",
    y_axis_label="Observed arrivals per hour",
)
p2.vbar(x=centers, top=counts, width=0.85, alpha=0.65)

show(column(p1, p2))

### Important interpretation

The lower chart need **not** look exactly like the intensity curve.

Why?

Because $\lambda(t)$ describes **expected local arrival intensity**, while one simulated day is still noisy.

If we averaged hundreds or thousands of simulated days, the average hourly count pattern would increasingly resemble the integrated intensity over each hour.

# 5. Compound Poisson processes

Counting events is sometimes not enough.

Suppose insurance claims arrive according to a Poisson process $N(t)$, while claim amounts are

$$
Y_1,Y_2,\ldots
$$

Then total loss by time $t$ is

$$
X(t)=\sum_{i=1}^{N(t)}Y_i.
$$

This is a **compound Poisson process**.

It separates two sources of randomness:

1. **frequency risk** — how many events occur;
2. **severity risk** — how large each event is.

If $E[Y]=\mu_Y$ and $\operatorname{Var}(Y)=\sigma_Y^2$, then

$$
E[X(t)]=\lambda t\mu_Y,
$$

and

$$
\operatorname{Var}(X(t))
=
\lambda t(\sigma_Y^2+\mu_Y^2).
$$

In [45]:
def simulate_compound_poisson(
    rate: float,
    horizon: float,
    severity_sampler: Callable[[int], np.ndarray],
    n_paths: int,
    rng: np.random.Generator,
) -> np.ndarray:
    totals = np.empty(n_paths)

    for i in range(n_paths):
        n_events = rng.poisson(rate * horizon)
        totals[i] = 0.0 if n_events == 0 else severity_sampler(n_events).sum()

    return totals


rate = 2.0
horizon = 12.0

# Example claim severity: Gamma(shape=2, scale=750)
shape, scale = 2.0, 750.0
severity_mean = shape * scale
severity_var = shape * scale**2

totals = simulate_compound_poisson(
    rate,
    horizon,
    severity_sampler=lambda n: rng.gamma(shape=shape, scale=scale, size=n),
    n_paths=40_000,
    rng=rng,
)

theoretical_mean = rate * horizon * severity_mean
theoretical_var = rate * horizon * (severity_var + severity_mean**2)

summary = pd.DataFrame({
    "quantity": ["mean", "variance"],
    "simulation": [totals.mean(), totals.var(ddof=1)],
    "theory": [theoretical_mean, theoretical_var],
})

display(summary)

,quantity,simulation,theory
0,mean,3.5952e+04,3.6000e+04
1,variance,8.1013e+07,8.1000e+07


In [46]:
hist, edges = np.histogram(totals, bins=60, density=True)
centers = (edges[:-1] + edges[1:]) / 2

p = figure(
    title="Distribution of aggregate loss in a compound Poisson model",
    width=800,
    height=420,
    x_axis_label="Total loss",
    y_axis_label="Estimated density",
)

p.vbar(
    x=centers,
    top=hist,
    width=np.diff(edges)*0.92,
    alpha=0.6,
)

show(p)

### Why the aggregate distribution is not simply Poisson

The number of claims is Poisson, but the **sum of random claim sizes** is generally continuous and may be strongly skewed.

This distinction is important in:

- insurance,
- operational risk,
- transaction-value modelling,
- network packet-size modelling,
- reliability cost modelling.

# 6. Continuous-time Markov chains

A CTMC has:

- continuous time $t\ge 0$;
- a discrete state space;
- the Markov property.

The future depends on the current state rather than the full past.

For small $h$,

$$
P(X(t+h)=j\mid X(t)=i)
=
q_{ij}h+o(h),
\qquad i\ne j.
$$

The constants $q_{ij}$ are **transition rates**, not probabilities.

## 6.1 The generator matrix

The generator is

$$
Q=(q_{ij}).
$$

For $i\ne j$,

$$
q_{ij}\ge 0.
$$

The diagonal is defined by

$$
q_{ii}=-\sum_{j\ne i}q_{ij},
$$

so every row sums to zero.

If the process is in state $i$, the total departure rate is

$$
\nu_i=-q_{ii}.
$$

Hence the holding time in state $i$ is

$$
H_i\sim\operatorname{Exponential}(\nu_i).
$$

Conditional on leaving state $i$,

$$
P(i\to j)
=
\frac{q_{ij}}{\nu_i}.
$$

This decomposes a CTMC into:

> **When do we leave?** + **Where do we go next?**

## 6.2 Example: a repairable machine

States:

- 0 = operational
- 1 = failed

Suppose failure occurs at rate $\lambda$ and repair occurs at rate $\mu$.

Then

$$
Q=
\begin{pmatrix}
-\lambda & \lambda\\
\mu & -\mu
\end{pmatrix}.
$$

The transition matrix at time $t$ is

$$
P(t)=e^{Qt}.
$$

In [47]:
failure_rate = 0.25
repair_rate = 1.0

Q = np.array([
    [-failure_rate, failure_rate],
    [repair_rate, -repair_rate],
])

print("Generator Q:")
display(pd.DataFrame(Q, index=["working", "failed"], columns=["working", "failed"]))

print("Row sums:", Q.sum(axis=1))

Generator Q:


,working,failed
working,-0.25,0.25
failed,1.00,-1.00


Row sums: [0. 0.]


In [48]:
times = np.linspace(0, 15, 300)

p_work_if_work = []
p_failed_if_work = []

for t in times:
    Pt = expm(Q * t)
    p_work_if_work.append(Pt[0, 0])
    p_failed_if_work.append(Pt[0, 1])

p = line_plot(
    times,
    {
        "P(working at t | working at 0)": np.array(p_work_if_work),
        "P(failed at t | working at 0)": np.array(p_failed_if_work),
    },
    title="Transient probabilities from the CTMC matrix exponential",
    x_label="Time",
    y_label="Probability",
    legend_location="center_right",
)

show(p)

### Long-run equilibrium

A stationary distribution $\pi$ satisfies

$$
\pi Q=0,
$$

with

$$
\sum_i \pi_i=1.
$$

For the two-state repairable machine,

$$
\pi_{working}
=
\frac{\mu}{\lambda+\mu},
$$

and

$$
\pi_{failed}
=
\frac{\lambda}{\lambda+\mu}.
$$

The first quantity is also the machine's long-run **availability**.

In [49]:
pi_work = repair_rate / (failure_rate + repair_rate)
pi_fail = failure_rate / (failure_rate + repair_rate)

print(f"Long-run working probability: {pi_work:.4f}")
print(f"Long-run failed probability:  {pi_fail:.4f}")

Long-run working probability: 0.8000
Long-run failed probability:  0.2000


## 6.3 Direct simulation of a general finite-state CTMC

The simulation algorithm is sometimes called a **jump-chain** or Gillespie-style simulation.

At state $i$:

1. compute departure rate $\nu_i=-q_{ii}$;
2. sample holding time $H\sim\operatorname{Exp}(\nu_i)$;
3. choose the next state with probabilities $q_{ij}/\nu_i$;
4. repeat.

If the realised path contains $m$ jumps, computational cost is roughly $\Theta(mk)$ for a naive choice among $k$ states, or better with sparse transition structures.

In [50]:
@dataclass
class CTMCSimulator:
    Q: np.ndarray

    def simulate(
        self,
        initial_state: int,
        horizon: float,
        rng: np.random.Generator,
    ) -> tuple[np.ndarray, np.ndarray]:
        Q = np.asarray(self.Q, dtype=float)
        state = int(initial_state)
        t = 0.0

        times = [t]
        states = [state]

        while t < horizon:
            departure_rate = -Q[state, state]

            if departure_rate <= 0:
                break

            holding = rng.exponential(1.0 / departure_rate)
            next_t = t + holding

            if next_t > horizon:
                break

            probs = Q[state].copy()
            probs[state] = 0.0
            probs = probs / departure_rate

            state = rng.choice(len(probs), p=probs)
            t = next_t

            times.append(t)
            states.append(state)

        return np.asarray(times), np.asarray(states)

In [51]:
simulator = CTMCSimulator(Q)
jump_times, states = simulator.simulate(0, horizon=20, rng=rng)

# Build step representation
x = []
y = []

for i in range(len(jump_times)):
    t0 = jump_times[i]
    t1 = jump_times[i+1] if i+1 < len(jump_times) else 20
    x.extend([t0, t1])
    y.extend([states[i], states[i]])

p = figure(
    title="One sample path of the repairable-machine CTMC",
    width=800,
    height=330,
    x_axis_label="Time",
    y_axis_label="State (0=working, 1=failed)",
    y_range=(-0.2, 1.2),
)

p.line(x, y, line_width=3)
p.scatter(jump_times, states, size=7)

show(p)

# 7. Kolmogorov forward equations

For a finite-state CTMC,

$$
P(t)=e^{Qt}.
$$

The transition matrix satisfies

$$
\frac{dP(t)}{dt}=P(t)Q.
$$

This is the **Kolmogorov forward equation**.

The backward equation is

$$
\frac{dP(t)}{dt}=QP(t).
$$

These equations are important because they convert probabilistic evolution into a system of differential equations.

That creates a bridge between:

- probability,
- linear algebra,
- dynamical systems,
- numerical analysis.

# 8. Birth-death processes

A birth-death process moves only between neighbouring integer states:

$$
n\to n+1
$$

at birth rate $\lambda_n$, and

$$
n\to n-1
$$

at death rate $\mu_n$.

Graphically:

$$
0
\rightleftarrows
1
\rightleftarrows
2
\rightleftarrows
3
\rightleftarrows\cdots
$$

Many systems naturally have this structure:

- queue length,
- population size,
- number of active jobs,
- number of infected individuals,
- number of functioning components.

# 9. The $M/M/1$ queue

An $M/M/1$ queue has:

- Poisson arrivals at rate $\lambda$;
- exponential service times at rate $\mu$;
- one server.

If $X(t)$ is the number of customers in the system, then

$$
n\to n+1
$$

at rate $\lambda$ and, for $n>0$,

$$
n\to n-1
$$

at rate $\mu$.

The utilisation is

$$
\rho=\frac{\lambda}{\mu}.
$$

A stationary distribution exists only when

$$
\rho<1.
$$

Then

$$
\pi_n=(1-\rho)\rho^n.
$$

## 9.1 Queue performance formulas

For $\rho<1$,

$$
L=\frac{\rho}{1-\rho}
$$

is the expected number in the system.

Using Little's law,

$$
L=\lambda W,
$$

we obtain

$$
W=\frac{1}{\mu-\lambda}.
$$

Similarly,

$$
L_q=\frac{\rho^2}{1-\rho},
$$

and

$$
W_q=\frac{\rho}{\mu-\lambda}.
$$

The most important insight is not the formulas themselves.

It is the **nonlinear explosion** as $\rho\to 1$.

In [52]:
mu = 10.0
rho_grid = np.linspace(0.05, 0.99, 300)
lambda_grid = rho_grid * mu

L = rho_grid / (1 - rho_grid)
W = 1 / (mu - lambda_grid)
Lq = rho_grid**2 / (1 - rho_grid)
Wq = rho_grid / (mu - lambda_grid)

p1 = line_plot(
    rho_grid,
    {
        "L: expected number in system": L,
        "Lq: expected number waiting": Lq,
    },
    title="Queue size explodes as utilisation approaches 1",
    x_label="Utilisation ρ",
    y_label="Expected customers",
)

p2 = line_plot(
    rho_grid,
    {
        "W: expected system time": W,
        "Wq: expected waiting time": Wq,
    },
    title="Waiting time explodes near capacity",
    x_label="Utilisation ρ",
    y_label="Expected time",
)

show(column(p1, p2))

### Systems-engineering interpretation

This is a powerful lesson for production systems:

> **A server operating at 99% average utilisation is not merely 10% more loaded than one at 90%. Its queueing delay can be dramatically worse.**

This is why capacity planning needs headroom.

The same intuition applies to:

- API services,
- ETL worker pools,
- database connections,
- hospital service capacity,
- call centres,
- manufacturing stations.

## 9.2 Simulating an $M/M/1$ queue as a birth-death process

In [53]:
def simulate_mm1_queue(
    arrival_rate: float,
    service_rate: float,
    horizon: float,
    rng: np.random.Generator,
):
    t = 0.0
    n = 0

    times = [0.0]
    states = [0]

    while t < horizon:
        birth = arrival_rate
        death = service_rate if n > 0 else 0.0
        total = birth + death

        if total <= 0:
            break

        dt = rng.exponential(1.0 / total)
        t_next = t + dt

        if t_next > horizon:
            break

        if rng.uniform() < birth / total:
            n += 1
        else:
            n -= 1

        t = t_next
        times.append(t)
        states.append(n)

    return np.asarray(times), np.asarray(states)


arrival_rate = 8.0
service_rate = 10.0

times_q, queue_states = simulate_mm1_queue(
    arrival_rate,
    service_rate,
    horizon=50,
    rng=rng,
)

p = figure(
    title="Sample path of an M/M/1 queue",
    width=800,
    height=370,
    x_axis_label="Time",
    y_axis_label="Customers in system",
)

p.step(times_q, queue_states, mode="after", line_width=2.2)
show(p)

## 9.3 Monte Carlo versus stationary queue distribution

We now sample the queue length at a large time across many independent simulations and compare the empirical distribution with

$$
\pi_n=(1-\rho)\rho^n.
$$

In [54]:
def mm1_state_at_horizon(
    arrival_rate: float,
    service_rate: float,
    horizon: float,
    rng: np.random.Generator,
) -> int:
    times, states = simulate_mm1_queue(
        arrival_rate,
        service_rate,
        horizon,
        rng,
    )
    return int(states[-1])


arrival_rate = 8.0
service_rate = 10.0
rho = arrival_rate / service_rate

n_sim = 3_000
terminal_states = np.array([
    mm1_state_at_horizon(arrival_rate, service_rate, 100.0, rng)
    for _ in range(n_sim)
])

max_state = min(30, int(np.quantile(terminal_states, 0.995)) + 2)
n_values = np.arange(max_state + 1)

empirical = np.array([(terminal_states == n).mean() for n in n_values])
stationary = (1-rho) * rho**n_values

source = ColumnDataSource({
    "n": n_values,
    "empirical": empirical,
    "stationary": stationary,
})

p = figure(
    title="M/M/1 terminal-state distribution versus stationary theory",
    width=800,
    height=420,
    x_axis_label="Queue state n",
    y_axis_label="Probability",
)

p.vbar(x="n", top="empirical", source=source, width=0.75, alpha=0.45, legend_label="Monte Carlo")
p.line(x="n", y="stationary", source=source, line_width=3, color=PALETTE[1], legend_label="Stationary")
p.scatter(x="n", y="stationary", source=source, size=8, color=PALETTE[1])
p.legend.location = "top_right"

show(p)

# 10. Multi-server queues: $M/M/c$

If there are $c$ identical servers, the departure rate depends on how many customers are currently being served.

If state $n$ means $n$ customers in the system,

$$
\mu_n=
\begin{cases}
n\mu, & n<c,\\
c\mu, & n\ge c.
\end{cases}
$$

This remains a birth-death process.

The multi-server setting is highly relevant for:

- call centres,
- hospital beds,
- thread pools,
- cloud workers,
- parallel ETL executors.

In [55]:
def mmc_stationary_probs(
    arrival_rate: float,
    service_rate: float,
    servers: int,
    max_n: int = 40,
) -> np.ndarray:
    c = servers
    a = arrival_rate / service_rate
    rho = arrival_rate / (c * service_rate)

    if rho >= 1:
        raise ValueError("Stationary distribution requires λ < cμ.")

    terms = [a**n / math.factorial(n) for n in range(c)]
    tail_base = a**c / math.factorial(c)
    p0_inv = sum(terms) + tail_base / (1-rho)
    p0 = 1 / p0_inv

    probs = np.zeros(max_n + 1)

    for n in range(max_n + 1):
        if n < c:
            probs[n] = p0 * a**n / math.factorial(n)
        else:
            probs[n] = p0 * a**n / (math.factorial(c) * c**(n-c))

    return probs


arrival_rate = 16.0
service_rate = 5.0
servers_list = [4, 5, 6]

n_values = np.arange(0, 31)
series = {}

for c in servers_list:
    probs = mmc_stationary_probs(arrival_rate, service_rate, c, max_n=30)
    series[f"c={c} servers"] = probs

p = line_plot(
    n_values,
    series,
    title="Stationary queue-size distributions under different server counts",
    x_label="Number in system",
    y_label="Probability",
)

show(p)

### Interpretation

Adding servers does more than reduce the average queue linearly.

Because queueing is nonlinear, a modest amount of extra capacity can sharply reduce the probability of very large backlogs.

# 11. Stochastic epidemic modelling

A stochastic SIR model tracks:

- $S(t)$ = susceptible population,
- $I(t)$ = infected population,
- $R(t)$ = recovered population.

Two events can occur:

### Infection

$$
(S,I,R)
	o
(S-1,I+1,R)
$$

with rate

$$
\beta \frac{SI}{N}.
$$

### Recovery

$$
(S,I,R)
	o
(S,I-1,R+1)
$$

with rate

$$
\gamma I.
$$

This is a CTMC because the next event time is exponentially distributed conditional on the current state.

In [56]:
@dataclass
class SIRCTMC:
    beta: float
    gamma: float
    population: int

    def simulate(
        self,
        S0: int,
        I0: int,
        R0: int,
        horizon: float,
        rng: np.random.Generator,
    ):
        t = 0.0
        S, I, R = S0, I0, R0

        times = [t]
        S_hist = [S]
        I_hist = [I]
        R_hist = [R]

        while t < horizon and I > 0:
            infection_rate = self.beta * S * I / self.population
            recovery_rate = self.gamma * I
            total_rate = infection_rate + recovery_rate

            if total_rate <= 0:
                break

            dt = rng.exponential(1.0 / total_rate)
            t_next = t + dt

            if t_next > horizon:
                break

            if rng.uniform() < infection_rate / total_rate:
                if S > 0:
                    S -= 1
                    I += 1
            else:
                I -= 1
                R += 1

            t = t_next
            times.append(t)
            S_hist.append(S)
            I_hist.append(I)
            R_hist.append(R)

        return (
            np.asarray(times),
            np.asarray(S_hist),
            np.asarray(I_hist),
            np.asarray(R_hist),
        )

In [57]:
N = 500
model = SIRCTMC(beta=0.45, gamma=0.15, population=N)

times_sir, S, I, R = model.simulate(
    S0=N-5,
    I0=5,
    R0=0,
    horizon=100,
    rng=rng,
)

p = line_plot(
    times_sir,
    {
        "Susceptible": S,
        "Infected": I,
        "Recovered": R,
    },
    title="One stochastic SIR epidemic trajectory",
    x_label="Time",
    y_label="People",
    legend_location="center_right",
)

show(p)

## 11.1 Why stochastic epidemics differ from deterministic ODEs

A deterministic SIR model produces one smooth trajectory for fixed parameters.

The stochastic CTMC produces **different trajectories on different runs**.

This matters especially when the number infected is small:

- one early recovery may extinguish the outbreak;
- several early infections may trigger a major epidemic.

So randomness can qualitatively change outcomes, not merely add visual noise.

In [58]:
n_paths = 12
p = figure(
    title="Multiple stochastic epidemic trajectories: infected population",
    width=800,
    height=420,
    x_axis_label="Time",
    y_axis_label="Infected",
)

for j in range(n_paths):
    t, _, I_path, _ = model.simulate(
        S0=N-5,
        I0=5,
        R0=0,
        horizon=80,
        rng=rng,
    )
    p.line(t, I_path, line_width=1.7, alpha=0.65, color=PALETTE[j % len(PALETTE)])

show(p)

# 12. Renewal processes

A Poisson process assumes exponential inter-arrival times.

A renewal process generalises this by allowing

$$
T_1,T_2,\ldots
$$

to be IID from **any positive distribution**.

Define

$$
S_n=T_1+\cdots+T_n
$$

and

$$
N(t)=\max\{n:S_n\le t\}.
$$

Every arrival is interpreted as a **renewal**.

Examples:

- replacing a failed component;
- machine maintenance cycles;
- customer repurchase intervals;
- repeated disease recurrence;
- inventory replenishment cycles.

## 12.1 Comparing Poisson and non-Poisson renewal paths

For exponential inter-arrivals, the coefficient of variation is

$$
CV=1.
$$

Other distributions allow:

- $CV<1$: more regular than Poisson;
- $CV>1$: more bursty than Poisson.

This gives renewal processes much greater modelling flexibility.

In [81]:
def simulate_renewal_arrivals(
    sampler: Callable[[], float],
    horizon: float,
) -> np.ndarray:
    arrivals = []
    t = 0.0

    while True:
        t += float(sampler())
        if t > horizon:
            break
        arrivals.append(t)

    return np.asarray(arrivals)


horizon = 20.0
mean_gap = 1.0

arr_exp = simulate_renewal_arrivals(
    lambda: rng.exponential(mean_gap),
    horizon,
)

# Gamma(shape=4, scale=0.25) also has mean 1, but is much more regular.
arr_gamma = simulate_renewal_arrivals(
    lambda: rng.gamma(shape=4.0, scale=0.25),
    horizon,
)

# Lognormal with approximately mean 1 and high variability.
sigma_ln = 1.0
mu_ln = -0.5 * sigma_ln**2
arr_lognormal = simulate_renewal_arrivals(
    lambda: rng.lognormal(mean=mu_ln, sigma=sigma_ln),
    horizon,
)

def arrival_step(arrivals, horizon):
    x = [0.0]
    y = [0]
    count = 0

    for t in arrivals:
        x.extend([t, t])
        y.extend([count, count+1])
        count += 1

    x.append(horizon)
    y.append(count)
    return np.asarray(x), np.asarray(y)

p = figure(
    title="Renewal-process paths with equal mean inter-arrival time but different variability",
    width=800,
    height=420,
    x_axis_label="Time",
    y_axis_label="Renewal count",
)

for idx, (label, arr) in enumerate({
    "Exponential": arr_exp,
    "Gamma": arr_gamma,
    "Lognormal": arr_lognormal,
}.items()):
    x, y = arrival_step(arr, horizon)
    p.line(x, y, line_width=2.2, color=PALETTE[idx], legend_label=label)

p.legend.location = "top_left"
show(p)

### Key observation

All three models may have the same **mean inter-arrival time**, yet their paths can look very different.

This is a central modelling lesson:

> Matching the mean is not enough. Variability and tail behaviour matter.

## 12.2 Elementary renewal theorem

If

$$
E[T_i]=\mu<\infty,
$$

then under standard conditions,

$$
\frac{N(t)}{t}
\to
\frac{1}{\mu}
$$

for large $t$.

Interpretation:

> Long-run renewal frequency is approximately the reciprocal of the mean cycle length.

In [80]:
def renewal_count_at_horizon(
    sampler: Callable[[], float],
    horizon: float,
) -> int:
    return len(simulate_renewal_arrivals(sampler, horizon))


horizons = np.linspace(10, 1000, 120)
mean_gap = 2.0

# One long renewal path, observed at increasing horizons
arrivals_long = simulate_renewal_arrivals(
    lambda: rng.gamma(shape=2.0, scale=mean_gap/2.0),
    horizon=float(horizons.max()),
)

rates = np.array([
    np.searchsorted(arrivals_long, h, side="right") / h
    for h in horizons
])

p = line_plot(
    horizons,
    {
        "Observed N(t)/t": rates,
        "Theoretical 1/E[T]": np.full_like(horizons, 1/mean_gap),
    },
    title="Elementary renewal theorem in one long simulation",
    x_label="t",
    y_label="Renewal rate N(t)/t",
    legend_location="top_right",
)

show(p)

# 13. Renewal-reward processes

Suppose each renewal cycle produces reward $R_i$ and has length $T_i$.

Then, under appropriate conditions,

$$
\text{long-run reward rate}
=
\frac{E[R]}{E[T]}.
$$

This is the renewal-reward theorem.

It is useful because many apparently complex systems can be decomposed into repeated cycles.

## 13.1 Preventive-maintenance example

Suppose a machine runs for a random lifetime $T$.

At failure:

- a repair cost is incurred;
- the machine is immediately renewed.

If every cycle produces operating revenue proportional to uptime and a repair cost at the end, the long-run economic rate can often be computed using

$$
\frac{E[R]}{E[T]}.
$$

In [61]:
n_cycles = 50_000

# Gamma lifetime with mean 100 hours
lifetimes = rng.gamma(shape=4.0, scale=25.0, size=n_cycles)

revenue_per_hour = 8.0
repair_cost = 250.0

cycle_rewards = revenue_per_hour * lifetimes - repair_cost

empirical_rate = cycle_rewards.sum() / lifetimes.sum()
theoretical_rate = (
    revenue_per_hour * lifetimes.mean() - repair_cost
) / lifetimes.mean()

summary = pd.DataFrame({
    "quantity": ["Empirical long-run reward rate", "Plug-in theoretical rate"],
    "value": [empirical_rate, theoretical_rate],
})

display(summary)

,quantity,value
0,Empirical long-run reward rate,5.4963
1,Plug-in theoretical rate,5.4963


# 14. Martingales

A process $(X_n)$ is a martingale with respect to information $(\mathcal F_n)$ if

$$
E[|X_n|]<\infty,
$$

and

$$
E[X_{n+1}\mid\mathcal F_n]=X_n.
$$

The intuition is a **fair game**:

> After accounting for all information currently available, the expected next value is the current value.

A submartingale has non-negative conditional drift:

$$
E[X_{n+1}\mid\mathcal F_n]\ge X_n,
$$

while a supermartingale has non-positive conditional drift:

$$
E[X_{n+1}\mid\mathcal F_n]\le X_n.
$$

## 14.1 Fair random walk as a martingale

Let

$$
X_n=\sum_{i=1}^n \xi_i,
$$

where

$$
P(\xi_i=1)=P(\xi_i=-1)=\frac12.
$$

Then

$$
E[\xi_{n+1}\mid\mathcal F_n]=0,
$$

so

$$
E[X_{n+1}\mid\mathcal F_n]=X_n.
$$

In [79]:
n_steps = 250
n_paths = 12

p = figure(
    title="Fair random walks: sample martingale paths",
    width=800,
    height=420,
    x_axis_label="Step n",
    y_axis_label="X_n",
)

for j in range(n_paths):
    increments = rng.choice([-1, 1], size=n_steps)
    path = np.r_[0, np.cumsum(increments)]
    p.line(
        np.arange(n_steps+1),
        path,
        line_width=1.6,
        alpha=0.65,
        color=PALETTE[j % len(PALETTE)],
    )

show(p)

## 14.2 Ensemble mean: martingale versus drift

A martingale does not mean each path remains flat.

Individual paths can wander far from zero.

The martingale property concerns **conditional expectation**, not path smoothness.

In [78]:
n_steps = 200
n_paths = 20_000

def simulate_biased_walk(p_up: float):
    increments = np.where(
        rng.uniform(size=(n_paths, n_steps)) < p_up,
        1,
        -1,
    )
    paths = np.c_[np.zeros(n_paths), np.cumsum(increments, axis=1)]
    return paths.mean(axis=0)

mean_fair = simulate_biased_walk(0.5)
mean_up = simulate_biased_walk(0.55)
mean_down = simulate_biased_walk(0.45)

steps = np.arange(n_steps+1)

p = line_plot(
    steps,
    {
        "Martingale p=0.50": mean_fair,
        "Submartingale-like p=0.55": mean_up,
        "Supermartingale-like p=0.45": mean_down,
    },
    title="Ensemble means reveal conditional drift",
    x_label="Step",
    y_label="Mean value across simulations",
)

show(p)

# 15. Stopping times and optional-stopping intuition

A stopping time $\tau$ is a random time whose occurrence can be determined using only information available up to that point.

Example:

> Stop the fair random walk the first time it reaches +10 or -10.

This is a valid stopping rule because at time $n$ we can determine whether the boundary has already been reached.

Under suitable regularity conditions, a martingale obeys versions of

$$
E[X_\tau]=E[X_0].
$$

The theorem has technical conditions because without them, pathological stopping rules can invalidate naive reasoning.

In [77]:
def stopped_fair_walk(
    lower: int,
    upper: int,
    max_steps: int,
    rng: np.random.Generator,
) -> tuple[int, int]:
    x = 0

    for step in range(1, max_steps + 1):
        x += 1 if rng.uniform() < 0.5 else -1

        if x <= lower or x >= upper:
            return x, step

    return x, max_steps


n_sim = 20_000
outcomes = np.empty(n_sim, dtype=int)
stopping_times = np.empty(n_sim, dtype=int)

for i in range(n_sim):
    outcomes[i], stopping_times[i] = stopped_fair_walk(
        lower=-10,
        upper=10,
        max_steps=20_000,
        rng=rng,
    )

print(f"Estimated E[X_tau]: {outcomes.mean():.4f}")
print(f"Mean stopping time: {stopping_times.mean():.2f}")
print(f"P(hit +10 first): {(outcomes == 10).mean():.4f}")

Estimated E[X_tau]: -0.0490
Mean stopping time: 100.23
P(hit +10 first): 0.4975


### Interpretation

For symmetric boundaries and a fair random walk, we expect approximately equal probability of hitting +10 or -10 first.

The simulation should produce

$$
E[X_\tau]\approx 0,
$$

consistent with the fair-game intuition.

# 16. Brownian motion

Brownian motion $(B_t)_{t\ge0}$ satisfies:

1. $B_0=0$;
2. it has independent increments;
3. for $0\le s<t$,

$$
B_t-B_s\sim N(0,t-s);
$$

4. sample paths are continuous.

Thus over a small interval $\Delta t$,

$$
\Delta B
\sim
N(0,\Delta t).
$$

Equivalently,

$$
\Delta B
=
\sqrt{\Delta t}Z,
\qquad Z\sim N(0,1).
$$

In [76]:
def simulate_brownian(
    horizon: float,
    n_steps: int,
    n_paths: int,
    rng: np.random.Generator,
):
    dt = horizon / n_steps
    increments = np.sqrt(dt) * rng.normal(size=(n_paths, n_steps))
    paths = np.c_[np.zeros(n_paths), np.cumsum(increments, axis=1)]
    times = np.linspace(0, horizon, n_steps + 1)
    return times, paths


times_bm, paths_bm = simulate_brownian(
    horizon=1.0,
    n_steps=1200,
    n_paths=10,
    rng=rng,
)

p = figure(
    title="Simulated Brownian-motion paths",
    width=800,
    height=420,
    x_axis_label="t",
    y_axis_label="B(t)",
)

for j in range(paths_bm.shape[0]):
    p.line(
        times_bm,
        paths_bm[j],
        line_width=1.6,
        alpha=0.7,
        color=PALETTE[j % len(PALETTE)],
    )

show(p)

## 16.1 Distribution at a fixed time

Brownian motion satisfies

$$
B_t\sim N(0,t).
$$

Therefore

$$
E[B_t]=0
$$

and

$$
\operatorname{Var}(B_t)=t.
$$

In [75]:
t_fixed = 2.5
n_sim = 60_000

Bt = np.sqrt(t_fixed) * rng.normal(size=n_sim)

hist, edges = np.histogram(Bt, bins=60, density=True)
centers = (edges[:-1] + edges[1:]) / 2

grid = np.linspace(Bt.min(), Bt.max(), 500)
exact = norm.pdf(grid, loc=0, scale=np.sqrt(t_fixed))

p = figure(
    title=f"Brownian motion marginal distribution at t={t_fixed}",
    width=800,
    height=420,
    x_axis_label="B(t)",
    y_axis_label="Density",
)

p.vbar(x=centers, top=hist, width=np.diff(edges)*0.92, alpha=0.45, legend_label="Simulation")
p.line(grid, exact, line_width=3, color=PALETTE[1], legend_label="N(0,t) density")
p.legend.location = "top_right"

show(p)

# 17. Brownian motion with drift

A common model is

$$
X_t=\mu t+\sigma B_t.
$$

Then

$$
X_t\sim N(\mu t,\sigma^2t).
$$

Interpretation:

- $\mu$ controls average drift;
- $\sigma$ controls random fluctuation.

This simple decomposition underlies many continuous-time models in finance, reliability and diffusion processes.

In [74]:
mu = 0.8
sigma = 1.2

times_bm, base_paths = simulate_brownian(
    horizon=3.0,
    n_steps=1200,
    n_paths=8,
    rng=rng,
)

drifted = mu * times_bm[None, :] + sigma * base_paths

p = figure(
    title="Brownian motion with drift",
    width=800,
    height=420,
    x_axis_label="t",
    y_axis_label="X(t)",
)

for j in range(drifted.shape[0]):
    p.line(
        times_bm,
        drifted[j],
        line_width=1.6,
        alpha=0.7,
        color=PALETTE[j % len(PALETTE)],
    )

p.line(
    times_bm,
    mu * times_bm,
    line_width=4,
    line_dash="dashed",
    color=PALETTE[9],
    legend_label="Mean path μt",
)

p.legend.location = "top_left"
show(p)

# 18. Quadratic variation — why Brownian motion is unusual

Brownian motion has extremely irregular paths.

Consider a partition

$$
0=t_0<t_1<\cdots<t_n=t.
$$

The quadratic variation is

$$
\sum_{i=1}^n
(B_{t_i}-B_{t_{i-1}})^2.
$$

As the partition becomes finer,

$$
\sum_{i=1}^n
(B_{t_i}-B_{t_{i-1}})^2
\to t
$$

in an appropriate probabilistic sense.

This property is fundamental to stochastic calculus.

Smooth deterministic functions instead have zero quadratic variation under standard conditions.

In [73]:
def brownian_qv_estimate(
    horizon: float,
    n_steps: int,
    rng: np.random.Generator,
):
    dt = horizon / n_steps
    increments = np.sqrt(dt) * rng.normal(size=n_steps)
    return np.sum(increments**2)


horizon = 2.0
partition_sizes = np.array([10, 20, 50, 100, 200, 500, 1000, 3000, 8000])
n_reps = 1000

qv_means = []

for n_steps in partition_sizes:
    estimates = np.array([
        brownian_qv_estimate(horizon, int(n_steps), rng)
        for _ in range(n_reps)
    ])
    qv_means.append(estimates.mean())

p = line_plot(
    partition_sizes,
    {
        "Monte Carlo mean quadratic variation": np.array(qv_means),
        "Theoretical limit t": np.full(len(partition_sizes), horizon),
    },
    title="Quadratic variation approaches the elapsed time",
    x_label="Number of partition intervals",
    y_label="Quadratic variation",
    legend_location="bottom_right",
)

p.xaxis.axis_label = "Number of partition intervals (finer partition →)"
show(p)

# 19. First-passage and hitting times

For a process $X_t$, a hitting time may be defined as

$$
\tau_a
=
\inf\{t\ge0:X_t\ge a\}.
$$

This same mathematical object appears in many applications:

- first time a queue exceeds capacity;
- first time an asset crosses a barrier;
- first time a reliability metric drops below a threshold;
- extinction time of an epidemic;
- ruin time of an insurer;
- first time Brownian motion reaches a level.

So first-passage analysis is one of the major unifying concepts in stochastic processes.

In [72]:
def brownian_first_passage(
    level: float,
    horizon: float,
    n_steps: int,
    rng: np.random.Generator,
):
    dt = horizon / n_steps
    increments = np.sqrt(dt) * rng.normal(size=n_steps)
    path = np.r_[0.0, np.cumsum(increments)]
    times = np.linspace(0, horizon, n_steps+1)

    hit_idx = np.flatnonzero(path >= level)

    if len(hit_idx) == 0:
        return np.nan

    return times[hit_idx[0]]


level = 1.0
n_sim = 5_000

hitting_times = np.array([
    brownian_first_passage(level, horizon=8.0, n_steps=1500, rng=rng)
    for _ in range(n_sim)
])

observed_hits = hitting_times[~np.isnan(hitting_times)]

hist, edges = np.histogram(observed_hits, bins=70, density=True)
centers = (edges[:-1] + edges[1:]) / 2

p = figure(
    title="Empirical Brownian first-passage times to level 1",
    width=800,
    height=420,
    x_axis_label="First-passage time",
    y_axis_label="Estimated density",
)

p.vbar(x=centers, top=hist, width=np.diff(edges)*0.92, alpha=0.6)
show(p)

print(f"Fraction hitting level {level} before horizon: {len(observed_hits)/n_sim:.4f}")

Fraction hitting level 1.0 before horizon: 0.7186


### Numerical subtlety

The simulation observes Brownian motion only on a finite grid.

A true Brownian path may cross the boundary between two grid points and then return below it before the next observation.

Therefore discrete simulation can **miss crossings**.

This is an important general lesson:

> In continuous-time stochastic models, discretisation error is different from Monte Carlo error.

- **Monte Carlo error** comes from using finitely many simulated paths.
- **Discretisation error** comes from approximating continuous time by a finite grid.

# 20. Integrated case study — modelling a cloud service

We now combine several ST4238 concepts into one systems-oriented story.

Suppose a cloud API receives requests.

### Model A: arrivals

Requests arrive at rate

$$
\lambda=45 	ext{ per minute}.
$$

### Model B: service

A server processes jobs at exponential rate

$$
\mu=12 	ext{ per minute}.
$$

### Model C: multiple workers

With $c$ workers, total maximum service capacity is

$$
c\mu.
$$

### Stability

We need

$$
\lambda<c\mu.
$$

For $c=4$,

$$
45<48,
$$

so the system is technically stable but highly utilised:

$$
\rho=\frac{45}{48}=0.9375.
$$

For $c=5$,

$$
\rho=\frac{45}{60}=0.75.
$$

The difference in queueing behaviour can be much larger than the raw capacity increase suggests.

In [71]:
arrival_rate = 45.0
service_rate = 12.0

rows = []

for c in range(4, 9):
    rho = arrival_rate / (c * service_rate)
    rows.append({
        "servers": c,
        "total_service_capacity": c * service_rate,
        "utilisation": rho,
        "stable": rho < 1,
    })

capacity_df = pd.DataFrame(rows)
display(capacity_df.round(4))
show(bokeh_table(capacity_df.round(4), height=210))

,servers,total_service_capacity,utilisation,stable
0,4,48.0,0.9375,True
1,5,60.0,0.7500,True
2,6,72.0,0.6250,True
3,7,84.0,0.5357,True
4,8,96.0,0.4688,True


## 20.1 Architecture lesson

Stochastic-process modelling gives a quantitative language for system-design questions:

- How much spare capacity is needed?
- How bursty are arrivals?
- Is Poisson traffic reasonable?
- How sensitive is latency to utilisation?
- Should we scale horizontally?
- What is the probability of hitting a dangerous backlog?
- How do failure and repair rates affect availability?

These are not just probability exercises; they are operational decision problems.

# 21. Common modelling mistakes

## Mistake 1 — treating a rate as a probability

$q_{ij}$ is an instantaneous transition rate, not necessarily less than 1.

For small $h$,

$$
P(i\to j	ext{ during }h)
\approx q_{ij}h.
$$

---

## Mistake 2 — assuming Poisson because data consists of counts

Count data is not automatically Poisson.

Poisson requires structural assumptions such as independent increments and an appropriate event-generation mechanism.

---

## Mistake 3 — checking only the mean

Two renewal processes can share the same mean inter-arrival time while having very different burstiness.

---

## Mistake 4 — ignoring stability conditions

Stationary queue formulas are meaningless when

$$
\rho\ge1.
$$

---

## Mistake 5 — assuming a martingale path is constant

A martingale is fair **in conditional expectation**. Individual paths can fluctuate dramatically.

---

## Mistake 6 — confusing Monte Carlo error and discretisation error

Increasing the number of paths reduces Monte Carlo noise.

Increasing time-grid resolution reduces discretisation error.

These solve different problems.

# 22. Computational complexity notes

For simulation-based work, algorithmic cost matters.

| Task | Approximate cost |
|---|---:|
| Poisson arrival simulation with $m$ events | $\Theta(m)$ |
| finite-state CTMC path with $m$ jumps and naive $k$-state selection | $O(mk)$ |
| sparse birth-death simulation with $m$ jumps | $\Theta(m)$ |
| Brownian simulation with $n$ steps and $p$ paths | $\Theta(np)$ |
| dense matrix exponential for $k\times k$ generator | roughly $O(k^3)$ |
| Monte Carlo with $M$ independent repetitions | linear in $M$ times cost per repetition |

### Efficiency tips

1. **Vectorise Brownian simulations** rather than looping over paths.
2. For sparse CTMCs, store only reachable transitions rather than scanning the full generator row.
3. Use analytic formulas when available and simulation as a validation tool.
4. Separate **model simulation** from **visualisation** so experiments can be rerun without duplicating plotting code.
5. Fix random seeds during debugging; vary them during robustness checks.
6. For very large Monte Carlo workloads, batch simulations to control memory usage.

# 23. Concept comparison table

| Model | State space | Time | Core assumption / mechanism | Typical use |
|---|---|---|---|---|
| Poisson process | counts | continuous | exponential gaps, constant rate | arrivals |
| NHPP | counts | continuous | time-varying intensity | demand patterns |
| Compound Poisson | aggregate value | continuous | Poisson arrivals + random marks | losses |
| CTMC | discrete states | continuous | exponential holding times | reliability |
| Birth-death | non-negative integers | continuous | nearest-neighbour jumps | queues |
| Renewal process | counts | continuous | IID positive cycle lengths | replacement |
| Martingale | general | discrete/continuous | zero conditional drift | fair games |
| Brownian motion | real-valued | continuous | Gaussian independent increments | diffusion |

# 24. Revision questions

Try these before opening the suggested solutions.

### Q1
Calls arrive according to a Poisson process with rate 8 per hour. What is the probability of exactly 3 calls in 30 minutes?

### Q2
In the same process, what is the expected waiting time until the next call?

### Q3
A rate-20 Poisson stream is independently labelled premium with probability 0.15. What is the premium-arrival rate?

### Q4
A repairable machine fails at rate $0.1$ and is repaired at rate $0.4$. What is its long-run availability?

### Q5
An $M/M/1$ queue has $\lambda=7$ and $\mu=10$. Compute $\rho$, $L$ and $W$.

### Q6
Why does an exponential holding-time model naturally produce a Markov process?

### Q7
What changes when a Poisson process is generalised into a renewal process?

### Q8
Why can a martingale still have very volatile sample paths?

### Q9
For Brownian motion, what is the distribution of $B_5-B_2$?

### Q10
What is the practical difference between Monte Carlo error and time-discretisation error?

# 25. Suggested solutions

### A1

Over 30 minutes,

$$
\lambda t=8\times 0.5=4.
$$

Therefore

$$
P(N=3)=e^{-4}\frac{4^3}{3!}.
$$

### A2

$$
E[T]=\frac{1}{8}\text{ hour}=7.5\text{ minutes}.
$$

### A3

By Poisson thinning,

$$
\lambda_{premium}=20(0.15)=3.
$$

### A4

$$
\pi_{working}
=
\frac{\mu}{\lambda+\mu}
=
\frac{0.4}{0.5}
=
0.8.
$$

### A5

$$
\rho=\frac{7}{10}=0.7.
$$

Then

$$
L=\frac{0.7}{0.3}=2.3333,
$$

and

$$
W=\frac{1}{10-7}=\frac13.
$$

### A6

The exponential distribution is memoryless. Once the current state is known, the residual holding-time distribution does not depend on how long the process has already remained there.

### A7

The exponential inter-arrival assumption is removed. Inter-arrival times may follow a general IID positive distribution.

### A8

The martingale property concerns conditional expectation, not path smoothness. Zero expected drift does not imply zero variance.

### A9

By independent stationary Gaussian increments,

$$
B_5-B_2\sim N(0,5-2)=N(0,3).
$$

### A10

Monte Carlo error comes from finitely many simulated paths. Discretisation error comes from approximating continuous time by a finite grid.

# 26. Further experiments

For deeper study, extend this notebook with:

1. **Poisson goodness-of-fit diagnostics** on real event-arrival data.
2. **Hawkes processes** for self-exciting event streams.
3. **CTMC parameter estimation** from observed transition data.
4. **Uniformisation** for CTMC transient probabilities.
5. **Gambler's ruin** solved analytically and by martingales.
6. **Reliability networks** with multiple component states.
7. **M/M/c waiting-time calculations** using Erlang-C.
8. **Competing-risks CTMCs**.
9. **Markov-modulated Poisson processes**.
10. **Brownian bridge** simulation.
11. **Geometric Brownian motion**.
12. **Euler-Maruyama** simulation for stochastic differential equations.
13. **Ornstein-Uhlenbeck processes** and mean reversion.
14. **Poisson random measures** and jump-diffusion models.

These are natural continuations because they reuse the same foundations:
event intensities, conditional independence, Markov structure, renewal ideas, martingales and continuous-time random motion.

# 27. Final mental model

If you remember only one modelling workflow, use this:

## Step 1 — define the state

$$
X(t)=\text{what completely summarises the system at time }t?
$$

## Step 2 — enumerate possible events

What can change the state?

## Step 3 — assign rates or waiting-time distributions

How quickly do those events occur?

## Step 4 — translate events into transitions

$$
\text{event}
\Rightarrow
\text{state update}.
$$

## Step 5 — determine the mathematical family

- Poisson?
- CTMC?
- birth-death?
- renewal?
- martingale?
- Brownian/diffusion?

## Step 6 — analyse

Compute:

- transient probabilities,
- stationary distributions,
- hitting probabilities,
- expected waiting times,
- long-run reward,
- extinction probabilities.

## Step 7 — simulate and validate

Use simulation to confirm intuition and explore cases where closed-form analysis is difficult.

---

The deepest lesson of ST4238 is that **randomness has structure**.

Once the state, event mechanism and time dynamics are chosen carefully, apparently complicated random systems can often be analysed using a surprisingly small collection of reusable stochastic-process ideas.

# 28. References and recommended reading

Useful references for deeper study include:

- Sheldon M. Ross — *Introduction to Probability Models*
- J. R. Norris — *Markov Chains*
- Geoffrey Grimmett & David Stirzaker — *Probability and Random Processes*
- Samuel Karlin & Howard Taylor — *A First Course in Stochastic Processes*
- NUS Department of Statistics and Data Science course information for the current module offering

For ST4238 revision, prioritise understanding the **model-building logic** before memorising formulas.